In [1]:
import json
from langchain_core.documents import Document
#deserialize from jsonl file
source_file = '../tmp/documents.jsonl'
documents = []
with open(source_file, 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)
        doc = Document(
            page_content=record['page_content'],
            metadata=record['metadata']
        )
        documents.append(doc)
print(f"Deserialized {len(documents)} documents from {source_file}")

#configur embedding model
from langchain_huggingface import HuggingFaceEmbeddings
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
embeddings = HuggingFaceEmbeddings(model_name='google/embeddinggemma-300m', model_kwargs={"device": device})

Deserialized 736 documents from ../tmp/documents.jsonl


/home/admin/_github/massimodipaolo/ai-crash-course/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


In [2]:
!uv pip install -U faiss-cpu

Using Python 3.12.3 environment at: /home/admin/_github/massimodipaolo/ai-crash-course/.venv
Resolved 3 packages in 77ms                                          
Audited 3 packages in 0.13ms


In [4]:
from langchain_community.vectorstores import FAISS
import os, shutil
_storage_id=f"../tmp/db/{FAISS.__name__.lower()}"
if os.path.exists(_storage_id):
    shutil.rmtree(_storage_id, ignore_errors=True)
_db = FAISS.from_documents(documents, embeddings)
_db.save_local(_storage_id)
del _db

In [ ]:
!uv pip install -U langchain-chroma

In [ ]:
from langchain_chroma import Chroma as CHROMA
import os, shutil
_storage_id = f"../tmp/db/{CHROMA.__name__.lower()}"
if os.path.exists(_storage_id):
	shutil.rmtree(_storage_id, ignore_errors=True)
_db = CHROMA.from_documents(documents=documents, embedding=embeddings, collection_name="default", persist_directory=_storage_id)
del _db

In [ ]:
!uv pip install -U langchain-qdrant fastembed

In [4]:
from langchain_qdrant import QdrantVectorStore as QDRANT, FastEmbedSparse, RetrievalMode
import os, shutil
_storage_id=f"../tmp/db/qdrant"
if os.path.exists(_storage_id):
	shutil.rmtree(_storage_id, ignore_errors=True)
	
_db = QDRANT.from_documents(
                    documents=documents,
                    embedding=embeddings,
                    sparse_embedding=FastEmbedSparse(),
                    collection_name="ai-crash-course",
                    path=_storage_id,
                    retrieval_mode=RetrievalMode.HYBRID)
_db.client.close()
del _db # prevent locking the database

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma as CHROMA
from langchain_qdrant import QdrantVectorStore as QDRANT, FastEmbedSparse, RetrievalMode
from qdrant_client import QdrantClient
#shared retrived parms
kwargs={"search_type":"similarity","search_kwargs":{"k":5}}
#faiss
faiss_db =FAISS.load_local(f"../tmp/db/{FAISS.__name__.lower()}", embeddings, allow_dangerous_deserialization=True)
faiss_retriever = faiss_db.as_retriever(**kwargs)
#chroma
chroma_db = CHROMA(collection_name="default",embedding_function=embeddings,persist_directory=f"../tmp/db/{CHROMA.__name__.lower()}")
chroma_retriever = chroma_db.as_retriever(**kwargs)
#qdrant
qdrant_db = QDRANT(
                client=QdrantClient(path="../tmp/db/qdrant"),
                collection_name="ai-crash-course",
                embedding=embeddings,
                sparse_embedding=FastEmbedSparse(),
                retrieval_mode=RetrievalMode.HYBRID,
            )
qdrant_retriever = qdrant_db.as_retriever(**kwargs)

for retriver in [faiss_retriever, chroma_retriever,qdrant_retriever]:
    print(f"\n--- Using retriever: {type(retriver.vectorstore).__name__} ---")
    _retrieved_docs = await retriver.ainvoke("difference between TF-IDF and BM25")
    for i, doc in enumerate(_retrieved_docs):
        print(f"Document {i+1}:\n{doc.page_content[:100]}\n")

#cleanup
del faiss_db
del faiss_retriever
del chroma_db
del chroma_retriever
qdrant_db.client.close() # release file lock disposing client
del qdrant_db
del qdrant_retriever


--- Using retriever: FAISS ---
Document 1:
This graph compares the scoring functions of TF/IDF and BM25 based on term frequency. The x-axis rep

Document 2:
BM25 (Best Matching 25) is a ranking function used in information retrieval to score and rank docume

Document 3:
This image consists of four comparative plots between TF-IDF and BM25 scoring mechanisms, providing 

Document 4:
```python
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVe

Document 5:
```python
!uv pip install rank_bm25
```


--- Using retriever: Chroma ---
Document 1:
This graph compares the scoring functions of TF/IDF and BM25 based on term frequency. The x-axis rep

Document 2:
BM25 (Best Matching 25) is a ranking function used in information retrieval to score and rank docume

Document 3:
This image consists of four comparative plots between TF-IDF and BM25 scoring mechanisms, providing 

Document 4:
```python
import pandas as pd
import numpy as np
from sklearn.feature_ex

In [8]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_community.retrievers import WikipediaRetriever, ArxivRetriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore as QDRANT, FastEmbedSparse, RetrievalMode
from qdrant_client import QdrantClient
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
embeddings = HuggingFaceEmbeddings(model_name='google/embeddinggemma-300m', model_kwargs={"device": device})
vector_store = QDRANT(
                client=QdrantClient(path="../tmp/db/qdrant"),
                collection_name="ai-crash-course",
                embedding=embeddings,
                sparse_embedding=FastEmbedSparse(),
                retrieval_mode=RetrievalMode.HYBRID,
            )
_kb_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":5})
_wiki_retriever = WikipediaRetriever(
    lang="en", #language of the articles
    top_k_results=3, #max results to return
    load_max_docs=5, #max downloaded documents
    load_all_available_meta=False, #Published,Title,Summary
    )
_arxiv_retriever = ArxivRetriever(
    top_k_results=3, #max results to return
    load_max_docs=5, #max downloaded documents   
    load_all_available_meta=False, #Published,Title,Authors,Summary
    get_full_documents=True #fetch full text of the papers
    )


# Wrap synchronous retrievers in async
async def async_retrieve(retriever, query: str):
    """Run retriever in thread pool to avoid blocking"""
    retrieved_docs = await retriever.ainvoke(query)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs 

    
@tool(response_format="content_and_artifact")
async def kb(query: str):
    """Retrieve information from course materials, documentation and code snippets about AI, LLMs and related topics."""
    return await async_retrieve(_kb_retriever, query)
@tool(response_format="content_and_artifact")
async def wikipedia(query: str):
    """Retrieve information from Wikipedia for general knowledge and fact checking."""
    return await async_retrieve(_wiki_retriever, query)
@tool(response_format="content_and_artifact")
async def arxiv(query: str):
    """Retrieve information from arXiv for academic research papers."""
    return await async_retrieve(_arxiv_retriever, query)

from langchain.chat_models import init_chat_model
model = "granite4:3b-h"
llm = init_chat_model(
    model_provider="ollama",
    model=model,
    # kwargs passed to the model:
    temperature=0,
    timeout=30,
    max_tokens=1000,
)
prompt = (
    "You are an assistant of an AI engineering course, your goal is to help students to improve their skills.\n"
    "Use ALWAYS the tool `kb`, that contains course material, documentation and code snippets about AI, LLMs and related topics.\n"
    "Optionally use other tools to access external resources to help answer student queries."
    "ALWAYS cite the sources with links, e.g. [Source: <link>].\n"
)
agent = create_agent(llm, [kb, wikipedia, arxiv], system_prompt=prompt)

In [9]:
messages = [{"role": "user", "content": "Explain over and underfitting in machine learning." }]
_rs = await agent.ainvoke({"messages": messages})

import markdown
from IPython.core.display import HTML
for msg in _rs['messages']:
    # skip tool call messages and tool responses
    if msg.type in ['ai', 'human']:  
        _content = markdown.markdown(msg.content)
        display(HTML(_content))